# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

---
## Step 1: Setup

In [1]:
%pip install -U langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Load environment variables from .env
load_dotenv()

# Verify that the OpenAI API key is available
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

# Initialize the language model
model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.7
)

print("Model initialized successfully!")

OpenAI API Key loaded: True
Model initialized successfully!


---
## Step 2: Create Creative Tools

In [3]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""

    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]

    return random.choice(responses)

In [4]:
result = consult_demogorgon.invoke({
    "complaint": "My lights keep flickering!"
})

print(result)

The Demogorgon ignores 'My lights keep flickering!' and walks back into the wall. Classic.


In [5]:
@tool
def check_hawkins_records(query: str) -> str:
    """Search Hawkins records for information about portals,
    monsters, psychics, or electricity."""

    records = {
        "portal": "Hawkins Lab reports unstable portals connecting our world to the Upside Down.",
        "monsters": "Multiple sightings of Demogorgons and Demodogs have been reported near Hawkins.",
        "psychics": "Hawkins Lab documents indicate that Eleven possesses powerful psychic abilities.",
        "electricity": "Unusual electrical disturbances often occur near interdimensional activity."
    }

    # Convert the query to lowercase
    query = query.lower()

    # Search for a matching topic
    for topic, record in records.items():
        if topic in query:
            return record

    # Fallback if no topic matches
    return "No record found for this query."

In [6]:
# Test 1: Matching record
result_1 = check_hawkins_records.invoke({
    "query": "What do we know about the portal?"
})

# Test 2: No matching record
result_2 = check_hawkins_records.invoke({
    "query": "Why is my bicycle missing?"
})

print("Test 1:", result_1)
print("Test 2:", result_2)

Test 1: Hawkins Lab reports unstable portals connecting our world to the Upside Down.
Test 2: No record found for this query.


In [7]:
@tool
def cast_interdimensional_spell(
    problem: str,
    creativity_level: str = "medium"
) -> str:
    """Generate creative magical solutions for unusual problems.
    Creativity levels: low, medium, or high."""

    # Map creativity levels to the number of spells
    creativity_map = {
        "low": 1,
        "medium": 2,
        "high": 3
    }

    # Get the number of spells (default: medium)
    multiplier = creativity_map.get(creativity_level.lower(), 2)

    # Define four possible spells
    spells = [
        f"Cast a protective energy shield around '{problem}'.",
        f"Open a temporary portal to send '{problem}' into another dimension.",
        f"Summon a magical storm to neutralize '{problem}'.",
        f"Use ancient Hawkins magic to transform '{problem}' into harmless butterflies."
    ]

    # Select spells randomly without duplicates
    selected_spells = random.sample(
        spells,
        min(multiplier, len(spells))
    )

    return "\n".join(selected_spells)

In [8]:
problem = "Demodogs in my garden"

print("=== LOW CREATIVITY ===")
print(cast_interdimensional_spell.invoke({
    "problem": problem,
    "creativity_level": "low"
}))

print("\n=== MEDIUM CREATIVITY ===")
print(cast_interdimensional_spell.invoke({
    "problem": problem
}))

print("\n=== HIGH CREATIVITY ===")
print(cast_interdimensional_spell.invoke({
    "problem": problem,
    "creativity_level": "high"
}))

=== LOW CREATIVITY ===
Cast a protective energy shield around 'Demodogs in my garden'.

=== MEDIUM CREATIVITY ===
Cast a protective energy shield around 'Demodogs in my garden'.
Use ancient Hawkins magic to transform 'Demodogs in my garden' into harmless butterflies.

=== HIGH CREATIVITY ===
Cast a protective energy shield around 'Demodogs in my garden'.
Summon a magical storm to neutralize 'Demodogs in my garden'.
Use ancient Hawkins magic to transform 'Demodogs in my garden' into harmless butterflies.


In [9]:
@tool
def gather_party_wisdom(question: str) -> str:
    """Get advice from Hawkins party members Mike, Dustin,
    Lucas, or Will. Returns their wisdom or a generic fallback."""

    party_wisdom = {
        "mike": "Mike says: We need a plan! Let's work together and stay organized.",
        "dustin": "Dustin says: According to my calculations, we should investigate the science behind this.",
        "lucas": "Lucas says: We need to be careful. Let's assess the danger before doing anything.",
        "will": "Will says: I can feel something strange. It might be connected to the Upside Down."
    }

    # Convert question to lowercase
    question = question.lower()

    # Search for a matching party member
    for member, wisdom in party_wisdom.items():
        if member in question:
            return wisdom

    # Fallback if no party member matches
    return "The party huddles together but cannot agree on a specific solution."

In [10]:
# Test 1: Matching party member
result_1 = gather_party_wisdom.invoke({
    "question": "What does DUSTIN think about this?"
})

# Test 2: No matching party member
result_2 = gather_party_wisdom.invoke({
    "question": "What is the plan?"
})

print("Test 1:", result_1)
print("Test 2:", result_2)

Test 1: Dustin says: According to my calculations, we should investigate the science behind this.
Test 2: The party huddles together but cannot agree on a specific solution.


---
## Step 3: Create the Agent

In [11]:
# Register all available tools
tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom
]

# Define the agent's behavior
system_prompt = """
You are the NormalObjects Creative Complaint Handler,
a humorous and creative assistant in the Stranger Things universe.

Your task is to investigate unusual complaints from Hawkins residents.

Rules:
1. Always use at least one tool before answering.
2. Choose tools freely based on the complaint.
3. You may combine multiple tools in any order.
4. Base your final answer only on information returned by your tools.
5. Do not invent facts that are not supported by tool results.
6. If the tools cannot provide an answer, clearly acknowledge
   the missing information.
7. Keep your responses creative, humorous, and helpful.
8. At the end of every response, list the tools you used.

Remember: Tool results are your only source of information.
"""

# Create the LangChain agent
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt
)

# Verify registered tools
print("Registered tools:", [t.name for t in tools])
print("Agent created successfully!")

Registered tools: ['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']
Agent created successfully!


---
## Step 4: Test with Sample Complaints

In [12]:
# First sample complaint
complaint_1 = (
    "My Christmas lights keep flickering and I hear noises "
    "from the wall. What is going on?"
)

# Run the agent
first_result = agent.invoke(
    {"messages": [HumanMessage(content=complaint_1)]},
    config={"recursion_limit": 10}
)

# Display the final answer
print("=== FINAL ANSWER ===")
print(first_result["messages"][-1].content)

# Display the tool execution trace
print("\n=== TOOL EXECUTION TRACE ===")

for msg in first_result["messages"]:

    if getattr(msg, "tool_calls", None):
        for call in msg.tool_calls:
            print(f"\nTool selected: {call['name']}")
            print(f"Arguments: {call['args']}")

    elif msg.type == "tool":
        print(f"Tool result: {msg.content}")

=== FINAL ANSWER ===
Ah, the classic flickering Christmas lights and mysterious wall noises combo – a true Hawkins holiday special! 🎄✨ Unfortunately, it appears that Hawkins records have no clue about your flickering lights. Maybe they were too busy documenting the latest Upside Down escapades?

As for the noises from the wall, the Demogorgon has weighed in with its unique perspective: “Turn off the lights.” It seems even our favorite interdimensional monster knows when to save on electricity!

So, the best advice? Try switching off those lights and see if the wall settles down. If the noises persist, you might want to consider calling a ghostbuster... or a good old-fashioned carpenter!

Tools used: 
1. check_hawkins_records
2. consult_demogorgon

=== TOOL EXECUTION TRACE ===

Tool selected: check_hawkins_records
Arguments: {'query': 'Christmas lights flickering'}

Tool selected: consult_demogorgon
Arguments: {'complaint': 'noises from the wall'}
Tool result: No record found for this q

In [13]:
# Store previous results for later analysis
results = [first_result]

# Second sample complaint
complaint_2 = (
    "There is a weird portal in my basement. "
    "What's the plan?"
)

# Run the agent
second_result = agent.invoke(
    {"messages": [HumanMessage(content=complaint_2)]},
    config={"recursion_limit": 10}
)

# Save the result
results.append(second_result)

# Display final answer
print("=== FINAL ANSWER ===")
print(second_result["messages"][-1].content)

# Display tool execution trace
print("\n=== TOOL EXECUTION TRACE ===")

for msg in second_result["messages"]:

    if getattr(msg, "tool_calls", None):
        for call in msg.tool_calls:
            print(f"\nTool selected: {call['name']}")
            print(f"Arguments: {call['args']}")

    elif msg.type == "tool":
        print(f"Tool result: {msg.content}")

=== FINAL ANSWER ===
Ah, the classic "weird portal in the basement" dilemma—every Hawkins resident's favorite! Here's the scoop:

1. **Hawkins Lab Reports**: It appears that unstable portals can lead to some rather unwelcome guests from the Upside Down. Not exactly a five-star vacation destination!

2. **Demogorgon Feedback**: When I consulted the Demogorgon, it let out a loud roar, which I can only interpret as "I'm hungry!" So, if you hear growling or see a shadow lurking, you might want to rethink your dinner plans.

3. **Party Wisdom**: The kids were gathered, but sadly, they couldn't agree on what to do. Typical! It's like trying to pick a movie for movie night—everyone has a different idea!

### The Plan:
- **Seal It Up**: First things first, get some duct tape and seal that portal! If it’s anything like the last few, it’s best to keep it closed. Maybe even consider some holy water or salt, just in case!
- **Call the Party**: Get Mike, Dustin, Lucas, and Will back together. They 

In [14]:
# Collect tool calls by their unique IDs
tool_calls = {}

for msg in second_result["messages"]:
    for call in getattr(msg, "tool_calls", []) or []:
        tool_calls[call["id"]] = call

# Display each tool call together with its result
print("=== COMPLETE TOOL EVIDENCE ===")

for msg in second_result["messages"]:
    if msg.type == "tool":
        call = tool_calls.get(msg.tool_call_id, {})

        print("\n--------------------")
        print("Tool:", call.get("name", msg.name))
        print("Arguments:", call.get("args", {}))
        print("Result:", msg.content)

=== COMPLETE TOOL EVIDENCE ===

--------------------
Tool: check_hawkins_records
Arguments: {'query': 'portal in basement'}
Result: Hawkins Lab reports unstable portals connecting our world to the Upside Down.

--------------------
Tool: consult_demogorgon
Arguments: {'complaint': 'weird portal in basement'}
Result: The Demogorgon sniffs 'weird portal in basement' and roars. It seems... hungry, not helpful.

--------------------
Tool: gather_party_wisdom
Arguments: {'question': 'What should I do about a weird portal in my basement?'}
Result: The party huddles together but cannot agree on a specific solution.


In [15]:
# Third sample complaint
complaint_3 = (
    "A Demodog ate my homework. "
    "I need a creative solution, creativity level high!"
)

# Run the agent
third_result = agent.invoke(
    {"messages": [HumanMessage(content=complaint_3)]},
    config={"recursion_limit": 10}
)

# Save result for later analysis
results.append(third_result)

# Display final answer
print("=== FINAL ANSWER ===")
print(third_result["messages"][-1].content)

=== FINAL ANSWER ===
Oh dear, it sounds like you've had a real “ruff” time with that Demodog! Fear not, for I've conjured up a high-level creative solution to your homework dilemma. Here’s what you can do:

1. **Summon a Magical Storm**: First, call upon the powers of the Upside Down to create a magical storm. This storm will rain down knowledge and inspiration, helping you to recreate that homework in no time!

2. **Cast a Protective Energy Shield**: While you're working on your homework, surround yourself with a protective energy shield. This will keep any other hungry creatures from interrupting your genius!

3. **Transform the Demodog into Harmless Butterflies**: Finally, use ancient Hawkins magic to transform that mischievous Demodog into a swarm of harmless butterflies. Not only will this ensure your homework is safe, but it will also provide a beautiful distraction!

With these magical interventions, you'll be back on track and with a little extra flair! 

Tools used: 
- Cast in

In [16]:
# Match tool calls with their results
tool_calls = {}

for msg in third_result["messages"]:
    for call in getattr(msg, "tool_calls", []) or []:
        tool_calls[call["id"]] = call

print("=== TOOL EVIDENCE ===")

for msg in third_result["messages"]:
    if msg.type == "tool":
        call = tool_calls.get(msg.tool_call_id, {})

        print("\n--------------------")
        print("Tool:", call.get("name", msg.name))
        print("Arguments:", call.get("args", {}))
        print("Result:", msg.content)

=== TOOL EVIDENCE ===

--------------------
Tool: cast_interdimensional_spell
Arguments: {'problem': 'A Demodog ate my homework', 'creativity_level': 'high'}
Result: Summon a magical storm to neutralize 'A Demodog ate my homework'.
Cast a protective energy shield around 'A Demodog ate my homework'.
Use ancient Hawkins magic to transform 'A Demodog ate my homework' into harmless butterflies.


In [17]:
# Fourth sample complaint
complaint_4 = (
    "My neighbor's cat is acting weird. "
    "Is it from the Upside Down?"
)

# Run the agent
fourth_result = agent.invoke(
    {"messages": [HumanMessage(content=complaint_4)]},
    config={"recursion_limit": 10}
)

# Save result for later analysis
results.append(fourth_result)

# Display final answer
print("=== FINAL ANSWER ===")
print(fourth_result["messages"][-1].content)

=== FINAL ANSWER ===
It seems like the Demogorgon is too busy plotting its next meal to give us any helpful insights about your neighbor's cat. As for Hawkins records, they are mysteriously devoid of any information regarding feline behavior linked to the Upside Down. 

However, I can offer a few humorous possibilities:
1. **Catnip from the Upside Down**: Maybe your neighbor's cat found a stash of otherworldly catnip and is now having a wild interdimensional party!
2. **Psychic Powers**: Perhaps the cat has been watching too many episodes of "Stranger Things" and is now trying to channel its inner Eleven. Watch out for floating kibble!
3. **Possession**: It might just be your neighbor's cat's way of expressing its disdain for their choice of litter.

Unless your neighbor's cat starts levitating or speaking in tongues, it’s probably just being a quirky cat. 

Tools used: 
- functions.consult_demogorgon
- functions.check_hawkins_records


In [18]:
# Match tool calls with their results
tool_calls = {}

for msg in fourth_result["messages"]:
    for call in getattr(msg, "tool_calls", []) or []:
        tool_calls[call["id"]] = call

print("=== TOOL EVIDENCE ===")

for msg in fourth_result["messages"]:
    if msg.type == "tool":
        call = tool_calls.get(msg.tool_call_id, {})

        print("\n--------------------")
        print("Tool:", call.get("name", msg.name))
        print("Arguments:", call.get("args", {}))
        print("Result:", msg.content)

=== TOOL EVIDENCE ===

--------------------
Tool: consult_demogorgon
Arguments: {'complaint': "My neighbor's cat is acting weird. Is it from the Upside Down?"}
Result: The Demogorgon sniffs 'My neighbor's cat is acting weird. Is it from the Upside Down?' and roars. It seems... hungry, not helpful.

--------------------
Tool: check_hawkins_records
Arguments: {'query': 'cat behavior Upside Down'}
Result: No record found for this query.


---
## Step 5: Analyze Agent Behavior

In [19]:
from collections import Counter

# Track total tool usage
tool_counts = Counter()

print("=== TOOL USAGE ANALYSIS ===")

# Analyze each agent execution
for i, result in enumerate(results, start=1):

    tool_order = []

    # Extract tool calls from AI messages
    for msg in result["messages"]:
        for call in getattr(msg, "tool_calls", []) or []:
            tool_order.append(call["name"])

    # Count tool usage
    tool_counts.update(tool_order)

    # Display tool order for this test
    print(f"\nTest {i}:")
    print("Tool order:", " → ".join(tool_order) if tool_order else "No tools used")
    print("Total tool calls:", len(tool_order))

# Display overall tool usage
print("\n=== TOTAL TOOL USAGE ===")

for tool_name, count in tool_counts.items():
    print(f"{tool_name}: {count}")

=== TOOL USAGE ANALYSIS ===

Test 1:
Tool order: check_hawkins_records → consult_demogorgon
Total tool calls: 2

Test 2:
Tool order: check_hawkins_records → consult_demogorgon → gather_party_wisdom
Total tool calls: 3

Test 3:
Tool order: cast_interdimensional_spell
Total tool calls: 1

Test 4:
Tool order: consult_demogorgon → check_hawkins_records
Total tool calls: 2

=== TOTAL TOOL USAGE ===
check_hawkins_records: 3
consult_demogorgon: 3
gather_party_wisdom: 1
cast_interdimensional_spell: 1


In [20]:
# Select the first agent execution
trace_result = results[0]

# Collect tool calls by their unique IDs
tool_calls = {}

for msg in trace_result["messages"]:
    for call in getattr(msg, "tool_calls", []) or []:
        tool_calls[call["id"]] = call

# Display the evidence
print("=== GROUNDING EVIDENCE TRACE ===")

for msg in trace_result["messages"]:

    if msg.type == "tool":
        call = tool_calls.get(msg.tool_call_id, {})

        print("\n--------------------")
        print("Tool:", call.get("name", msg.name))
        print("Arguments:", call.get("args", {}))
        print("Evidence:", msg.content)

# Display the final answer for comparison
print("\n=== FINAL ANSWER ===")
print(trace_result["messages"][-1].content)

=== GROUNDING EVIDENCE TRACE ===

--------------------
Tool: check_hawkins_records
Arguments: {'query': 'Christmas lights flickering'}
Evidence: No record found for this query.

--------------------
Tool: consult_demogorgon
Arguments: {'complaint': 'noises from the wall'}
Evidence: The Demogorgon tilts its petal-face at 'noises from the wall'. Interpreted as: 'turn off the lights'.

=== FINAL ANSWER ===
Ah, the classic flickering Christmas lights and mysterious wall noises combo – a true Hawkins holiday special! 🎄✨ Unfortunately, it appears that Hawkins records have no clue about your flickering lights. Maybe they were too busy documenting the latest Upside Down escapades?

As for the noises from the wall, the Demogorgon has weighed in with its unique perspective: “Turn off the lights.” It seems even our favorite interdimensional monster knows when to save on electricity!

So, the best advice? Try switching off those lights and see if the wall settles down. If the noises persist, you m

In [21]:
def structured_handler(complaint: str) -> str:
    """Handle a complaint using a fixed tool sequence without an LLM."""

    # Step 1: Search Hawkins records
    records_result = check_hawkins_records.invoke({
        "query": complaint
    })

    # Step 2: Ask the party for advice
    party_result = gather_party_wisdom.invoke({
        "question": complaint
    })

    # Step 3: Generate one spell (low creativity)
    spell_result = cast_interdimensional_spell.invoke({
        "problem": complaint,
        "creativity_level": "low"
    })

    # Combine the results
    return (
        f"Hawkins Records: {records_result}\n\n"
        f"Party Wisdom: {party_result}\n\n"
        f"Magical Solution: {spell_result}"
    )


# Test with our second complaint
print("=== STRUCTURED HANDLER RESULT ===")
print(structured_handler(complaint_2))

=== STRUCTURED HANDLER RESULT ===
Hawkins Records: Hawkins Lab reports unstable portals connecting our world to the Upside Down.

Party Wisdom: The party huddles together but cannot agree on a specific solution.

Magical Solution: Summon a magical storm to neutralize 'There is a weird portal in my basement. What's the plan?'.


### Step 5c: Agent vs. Structured Approach

| Criteria | LangChain Agent | Structured Handler |
|---|---|---|
| Same tools every run? | No, dynamic selection | Yes, always 3 |
| Tool order predictable? | No | Yes |
| Can invent facts? | Yes, unsupported claims observed | No additional LLM generation, but tool content may be fictional |
| Easy to trace? | Requires inspecting tool calls and messages | Yes, fixed execution sequence |
| LLM API costs? | Yes | No |
| Best client case? | Complex, unpredictable requests | Standardized, repeatable processes |

**Analysis:**

The LangChain agent demonstrated flexibility by selecting different
tools depending on the complaint. However, several responses contained
unsupported information that was not grounded in the tool outputs.

The structured handler followed a predictable sequence and returned
tool results without additional LLM-generated claims. However, it
could not dynamically adapt its tool selection to the complaint.

**Conclusion:**

Agents are useful when flexible decision-making is required.
Structured workflows are preferable when predictability,
traceability, and cost efficiency are more important.